# 04 - Test on images the model has never seen

The U-Net was trained on the training split, and every choice so far was made on the validation split. This notebook runs the same model, with the same weights and the same post-processing, on the 65 images of `stage1_test`. These images played no part in training or in any decision. The expert masks come from `stage1_solution.csv`, which was released after the 2018 Data Science Bowl.

The main question is H3b in `docs/plan.md`, written before this notebook was run: does the entropy over the nucleus area flag the worst images better than chance on new images? Nothing here is tuned.

Kaggle setup: Accelerator GPU T4, Internet on, and **+ Add Input > Your Work > the saved version of notebook 02**, so that `unet_best.pt` is available under `/kaggle/input`.
Output: `out/per_image_test.csv`, `out/by_type_test.csv`, `out/errors_by_type_test.csv`, `out/test_warning.json`, `out/worst_test_images.png`, zipped as `results_04.zip`.

In [ ]:
import glob, zipfile, urllib.request, json, shutil, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import io as skio
from skimage.segmentation import find_boundaries
import torch
import torch.nn as nn
import torch.nn.functional as F

WORK = Path('/kaggle/working') if Path('/kaggle').exists() else Path('.')
DATA = Path(os.environ.get('NUCLEI_DATA', '/tmp/nuclei_data'))   # raw data is not saved as output
OUT = WORK / 'out'
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

BASE = 'https://data.broadinstitute.org/bbbc/BBBC038/'
for name in ['stage1_test.zip', 'stage1_solution.csv']:
    if not (DATA / name).exists():
        urllib.request.urlretrieve(BASE + name, DATA / name)
test_dir = DATA / 'stage1_test'
if not test_dir.exists():
    with zipfile.ZipFile(DATA / 'stage1_test.zip') as z:
        z.extractall(test_dir)

solution = pd.read_csv(DATA / 'stage1_solution.csv')
if 'Usage' in solution.columns:
    solution = solution[solution.Usage != 'Ignored']
solution = solution.dropna(subset=['EncodedPixels'])
paths = {Path(p).stem: p for p in glob.glob(str(test_dir / '**' / 'images' / '*.png'), recursive=True)}
ids = sorted(set(paths) & set(solution.ImageId))
print(len(paths), 'test images,', len(ids), 'with expert masks,', len(solution), 'nuclei in the solution file')

The expert masks are stored as run-length codes: pairs of (start, length), pixels counted from 1, top to bottom and then left to right. The image type comes from the same rule as in notebook 01.

In [ ]:
def rle_decode(rle, h, w):
    s = np.asarray(str(rle).split(), dtype=np.int64)
    flat = np.zeros(h * w, bool)
    for start, length in zip(s[0::2] - 1, s[1::2]):
        flat[start:start + length] = True
    return flat.reshape(w, h).T


masks_of = {k: g.EncodedPixels.tolist() for k, g in solution.groupby('ImageId')}

def load(image_id):
    img = skio.imread(paths[image_id])
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    img = img[..., :3]
    h, w = img.shape[:2]
    label = np.zeros((h, w), np.int32)
    for k, rle in enumerate(masks_of[image_id], start=1):
        label[rle_decode(rle, h, w)] = k
    return img, label

def normalise(img):
    x = img.astype(np.float32) / 255.0
    return (x - x.mean()) / (x.std() + 1e-6)

def image_type(img):
    # same rule as notebook 01: colour stain -> histology; grey with dark background -> fluorescence; else brightfield
    x = img.astype(np.float32)
    colour = np.abs(x[..., 0] - x[..., 1]).mean() + np.abs(x[..., 1] - x[..., 2]).mean()
    if colour > 5:
        return 'histology'
    return 'fluorescence' if x.mean() < 100 else 'brightfield'

## The model and the scoring functions, unchanged from notebook 03

In [ ]:
# same model as in notebook 02 - the weights only load into exactly this architecture
class DoubleConv(nn.Module):
    """Conv 3x3 -> BatchNorm -> ReLU, two times. Height and width stay the same."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_ch=3, n_classes=3, base=32):
        super().__init__()
        chs = [base, base * 2, base * 4, base * 8]          # 32, 64, 128, 256
        self.downs = nn.ModuleList()
        c_in = in_ch
        for c in chs:                                        # encoder: DoubleConv, then MaxPool halves the size
            self.downs.append(DoubleConv(c_in, c))
            c_in = c
        self.pool = nn.MaxPool2d(2)
        self.bottom = DoubleConv(chs[-1], chs[-1] * 2)       # 256 -> 512 at 1/16 of the input size
        self.ups = nn.ModuleList()
        self.up_convs = nn.ModuleList()
        c_in = chs[-1] * 2
        for c in reversed(chs):                              # decoder: upsample, join the skip, DoubleConv
            self.ups.append(nn.ConvTranspose2d(c_in, c, kernel_size=2, stride=2))
            self.up_convs.append(DoubleConv(c * 2, c))
            c_in = c
        self.head = nn.Conv2d(chs[0], n_classes, kernel_size=1)   # 3 logits per pixel

    def forward(self, x):
        skips = []
        for down in self.downs:
            x = down(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottom(x)
        for up, conv, skip in zip(self.ups, self.up_convs, reversed(skips)):
            x = up(x)
            x = conv(torch.cat([skip, x], dim=1))
        return self.head(x)

In [ ]:
weights = sorted(glob.glob('/kaggle/input/**/unet_best.pt', recursive=True)) or sorted(glob.glob('**/unet_best.pt', recursive=True))
assert weights, 'unet_best.pt not found: add the output of notebook 02 with + Add Input'
print('weights:', weights[0])
model = UNet().to(DEVICE)
model.load_state_dict(torch.load(weights[0], map_location=DEVICE))
model.eval()

def pad16(x):
    h, w = x.shape[-2:]
    ph, pw = (-h) % 16, (-w) % 16
    return F.pad(x, (0, pw, 0, ph), mode='reflect'), h, w

@torch.no_grad()
def predict(img):
    x = torch.from_numpy(normalise(img).transpose(2, 0, 1))[None].to(DEVICE)
    xt, h, w = pad16(x)
    logits = model(xt)[..., :h, :w]
    return torch.softmax(logits, 1)[0].cpu().numpy()

In [ ]:
import numpy as np
from scipy import ndimage as ndi
from skimage.segmentation import watershed, relabel_sequential


def remove_small(label, min_size):
    """Drop labelled objects smaller than min_size pixels and renumber 1..n."""
    sizes = np.bincount(label.ravel())
    small = sizes < min_size
    small[0] = False
    label = label.copy()
    label[small[label]] = 0
    return relabel_sequential(label)[0]


def to_instances(prob, thr=0.5, min_size=15):
    """Pixel probabilities (3, H, W) -> instance labels (H, W).

    Seeds are the connected regions of 'interior'. Each seed grows by watershed
    over the whole nucleus area (interior + boundary), so touching nuclei that
    the boundary class separated stay separate.
    """
    interior = prob[1] > thr
    nucleus = (prob[1] + prob[2]) > thr
    seeds = remove_small(ndi.label(interior)[0], min_size // 3 + 1)
    label = watershed(-prob[1], seeds, mask=nucleus)
    return remove_small(label, min_size)


def match(true, pred):
    """IoU between every true and every predicted nucleus, plus the overlap counts."""
    true = relabel_sequential(true)[0]
    pred = relabel_sequential(pred)[0]
    nt, npred = int(true.max()), int(pred.max())
    inter = np.bincount(true.ravel().astype(np.int64) * (npred + 1) + pred.ravel(),
                        minlength=(nt + 1) * (npred + 1)).reshape(nt + 1, npred + 1)
    area_t = inter.sum(1)[1:]
    area_p = inter.sum(0)[1:]
    inter = inter[1:, 1:]
    union = area_t[:, None] + area_p[None, :] - inter
    iou = inter / np.maximum(union, 1)
    return iou, inter, area_t, area_p


def dsb_scores(iou, thresholds=np.arange(0.5, 1.0, 0.05)):
    """Data Science Bowl 2018 score for one image and F1 at IoU 0.5.

    Above IoU 0.5 a nucleus can match at most one prediction, so counting
    rows and columns with a match is the same as a one-to-one matching.
    """
    nt, npred = iou.shape
    per_t = []
    for t in thresholds:
        hit = iou > t
        tp = int(hit.any(1).sum())
        fn = nt - tp
        fp = npred - int(hit.any(0).sum())
        per_t.append(tp / (tp + fp + fn) if tp + fp + fn else 1.0)
    hit = iou > 0.5
    tp = int(hit.any(1).sum())
    fn, fp = nt - tp, npred - int(hit.any(0).sum())
    f1 = 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 1.0
    return float(np.mean(per_t)), f1, tp, fp, fn


def merges_and_splits(inter, area_t, area_p, share=0.5):
    """Merge: one prediction covers more than half of two or more true nuclei.
    Split: one true nucleus covers more than half of two or more predictions."""
    if inter.size == 0:
        return 0, 0
    cover_t = inter / np.maximum(area_t[:, None], 1)
    cover_p = inter / np.maximum(area_p[None, :], 1)
    merges = int(((cover_t > share).sum(0) >= 2).sum())
    splits = int(((cover_p > share).sum(1) >= 2).sum())
    return merges, splits


def entropy_map(prob):
    p = np.clip(prob, 1e-8, 1.0)
    return -(p * np.log(p)).sum(0)


def warning_test(uncertainty, score, share=0.1):
    """Flag the most uncertain images and count how many of the worst images are caught."""
    from scipy.stats import hypergeom, spearmanr
    n = len(score)
    k = max(1, int(np.ceil(share * n)))
    worst = set(np.argsort(score)[:k])
    flagged = set(np.argsort(-np.asarray(uncertainty))[:k])
    caught = len(worst & flagged)
    expected = k * k / n
    p_value = float(hypergeom.sf(caught - 1, n, k, k))
    rho, rho_p = spearmanr(uncertainty, score)
    return dict(n_images=n, k=k, caught=caught, expected_by_chance=round(expected, 2),
                p_value=p_value, spearman_rho=float(rho), spearman_p=float(rho_p),
                missed=sorted(int(i) for i in worst - flagged))


def error_types(iou, inter, area_t, area_p, share=0.5, iou_hit=0.5, iou_none=0.1):
    """Give every error at IoU 0.5 one cause, so the error counts add up.

    A true nucleus without a match (missed):
      merged      - more than half of it lies in a prediction that covers more than half of 2+ true nuclei
      cut         - it holds more than half of 2+ predictions (it was cut into pieces)
      not_found   - no prediction reaches IoU 0.1 with it
      bad_outline - everything else (found, but the outline is too far off)
    A prediction without a match (extra):
      merged      - it covers more than half of 2+ true nuclei
      cut         - more than half of it lies in a true nucleus that was cut into 2+ predictions
      false_alarm - it reaches IoU 0.1 with no true nucleus
      bad_outline - everything else
    """
    nt, npred = iou.shape
    out = {f'{s}_{c}': 0 for s in ('missed', 'extra') for c in ('merged', 'cut', 'not_found', 'false_alarm', 'bad_outline')}
    out = {k: v for k, v in out.items() if k not in ('missed_false_alarm', 'extra_not_found')}
    if nt == 0 or npred == 0:
        out['missed_not_found'] = nt
        out['extra_false_alarm'] = npred
        return out
    cover_t = inter / np.maximum(area_t[:, None], 1)   # share of true nucleus t inside prediction p
    cover_p = inter / np.maximum(area_p[None, :], 1)   # share of prediction p inside true nucleus t
    merge_pred = (cover_t > share).sum(0) >= 2
    cut_true = (cover_p > share).sum(1) >= 2
    hit = iou > iou_hit
    for t in np.where(~hit.any(1))[0]:
        if (merge_pred & (cover_t[t] > share)).any():
            out['missed_merged'] += 1
        elif cut_true[t]:
            out['missed_cut'] += 1
        elif iou[t].max() < iou_none:
            out['missed_not_found'] += 1
        else:
            out['missed_bad_outline'] += 1
    for p in np.where(~hit.any(0))[0]:
        if merge_pred[p]:
            out['extra_merged'] += 1
        elif (cut_true & (cover_p[:, p] > share)).any():
            out['extra_cut'] += 1
        elif iou[:, p].max() < iou_none:
            out['extra_false_alarm'] += 1
        else:
            out['extra_bad_outline'] += 1
    return out

In [ ]:
rows, keep = [], {}
for image_id in ids:
    img, true = load(image_id)
    prob = predict(img)
    pred = to_instances(prob)
    iou, inter, area_t, area_p = match(true, pred)
    ap, f1, tp, fp, fn = dsb_scores(iou)
    merges, splits = merges_and_splits(inter, area_t, area_p)
    causes = error_types(iou, inter, area_t, area_p)
    ent = entropy_map(prob)
    band = find_boundaries(pred > 0, mode='thick') | (pred > 0)
    rows.append(dict(id=image_id, type=image_type(img), n_true=int(true.max()), n_pred=int(pred.max()),
                     ap=ap, f1_50=f1, tp=tp, fp=fp, fn=fn, merges=merges, splits=splits,
                     entropy=float(ent.mean()),
                     entropy_nuclei=float(ent[band].mean()) if band.any() else 0.0, **causes))
    keep[image_id] = (img, true, pred)
df = pd.DataFrame(rows)
df.to_csv(OUT / 'per_image_test.csv', index=False)
print(len(df), 'test images, mean AP', round(df.ap.mean(), 3))
print(df.type.value_counts())

## H1 and H2 on the test images

In [ ]:
by_type = df.groupby('type').agg(images=('id', 'size'), mean_ap=('ap', 'mean'), median_ap=('ap', 'median'),
                                 mean_f1_50=('f1_50', 'mean'), nuclei=('n_true', 'sum'),
                                 merges=('merges', 'sum'), splits=('splits', 'sum'),
                                 missed=('fn', 'sum'), extra=('fp', 'sum')).round(3)
by_type.loc['all'] = [len(df), df.ap.mean(), df.ap.median(), df.f1_50.mean(), df.n_true.sum(),
                      df.merges.sum(), df.splits.sum(), df.fn.sum(), df.fp.sum()]
by_type = by_type.round(3)
by_type.to_csv(OUT / 'by_type_test.csv')
print(by_type.to_string())

causes = [c for c in df.columns if c.startswith(('missed_', 'extra_'))]
errors = df.groupby('type')[causes].sum()
errors.loc['all'] = df[causes].sum()
errors['all_errors'] = errors[causes].sum(1)
errors['merged_or_cut'] = errors[['missed_merged', 'missed_cut', 'extra_merged', 'extra_cut']].sum(1)
errors['share_merged_or_cut'] = (errors.merged_or_cut / errors.all_errors).round(3)
errors.to_csv(OUT / 'errors_by_type_test.csv')
print()
print(errors.T.to_string())

## H3b: does the nucleus-area entropy flag the worst test images?

Fixed before this run (`docs/plan.md`): the 10 % of test images with the highest entropy over the nucleus area are flagged, and the 10 % with the lowest AP are the worst. H3b holds if the overlap is larger than chance with p < 0.05 (one-sided hypergeometric test). The whole-image entropy, the measure that failed on the validation split, is shown next to it for comparison only.

In [ ]:
h3b = warning_test(df.entropy_nuclei.values, df.ap.values)
whole = warning_test(df.entropy.values, df.ap.values)
h3b['missed_ids'] = [df.id[i] for i in h3b['missed']]
result = dict(h3b_nucleus_area_entropy=h3b, h3b_holds=bool(h3b['p_value'] < 0.05),
              whole_image_entropy_for_comparison={k: v for k, v in whole.items() if k != 'missed'})
(OUT / 'test_warning.json').write_text(json.dumps(result, indent=2, default=float))
print(f"H3b: worst {h3b['k']} test images, {h3b['caught']} flagged by the nucleus-area entropy, "
      f"{h3b['expected_by_chance']} expected by chance (p = {h3b['p_value']:.3g}) -> "
      f"{'holds' if result['h3b_holds'] else 'does not hold'}")
print(f"Spearman rank correlation nucleus-area entropy vs AP: {h3b['spearman_rho']:.2f} (p = {h3b['spearman_p']:.3g})")
print(f"whole-image entropy, for comparison: {whole['caught']} caught, rho {whole['spearman_rho']:.2f}")

types = sorted(df.type.unique())
fig, ax = plt.subplots(figsize=(6, 4))
for t in types:
    s = df[df.type == t]
    ax.scatter(s.entropy_nuclei, s.ap, s=14, label=t)
ax.set_xlabel('mean entropy over the nucleus area'); ax.set_ylabel('AP per image')
ax.legend(); ax.set_title('Test images: uncertainty against score')
plt.tight_layout(); plt.savefig(OUT / 'uncertainty_vs_ap_test.png', dpi=120); plt.show()

## The worst test images

Green: expert outline. Red: predicted outline. The title says whether the nucleus-area entropy flagged the image.

In [ ]:
flagged = set(df.sort_values('entropy_nuclei', ascending=False).id[:h3b['k']])
worst = df.sort_values('ap').head(8)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax in axes.ravel():
    ax.axis('off')
for ax, r in zip(axes.ravel(), worst.itertuples()):
    img, true, pred = keep[r.id]
    show = img.astype(float) / max(img.max(), 1)
    show[find_boundaries(true)] = [0, 1, 0]
    show[find_boundaries(pred)] = [1, 0, 0]
    ax.imshow(show)
    ax.set_title(f"{r.type}, AP {r.ap:.2f}, {'flagged' if r.id in flagged else 'NOT flagged'}", fontsize=10)
plt.tight_layout(); plt.savefig(OUT / 'worst_test_images.png', dpi=110); plt.show()

## Answers

Write them here in your own words after looking at the numbers above:

- H1 on new images (histology and brightfield lower than fluorescence):
- H2 on new images (merged or cut nuclei more than half of the errors on fluorescence):
- H3b (nucleus-area entropy flags the worst test images better than chance):

In [ ]:
shutil.make_archive(str(WORK / 'results_04'), 'zip', OUT)
print(sorted(p.name for p in OUT.iterdir()))